# Mínimos cuadrados vs. descenso de gradiente: cuándo se usa cada uno

En [`00_introduccion_y_scikit_learn.ipynb`](00_introduccion_y_scikit_learn.ipynb)
viste que `LinearRegression` resuelve la regresión lineal simple con una
fórmula algebraica exacta (mínimos cuadrados), sin necesidad de descenso de
gradiente. En [`01_funcion_de_costo.ipynb`](01_funcion_de_costo.ipynb)
programaste ese descenso de gradiente a mano. Podría parecer que "mínimos
cuadrados" y "descenso de gradiente" son dos alternativas para resolver el
mismo problema (MSE), y que puedes elegir cualquiera. No es así: **la función
de costo (MSE) es solo la pregunta** — "¿qué tan mal está esta recta?".
Mínimos cuadrados y descenso de gradiente son dos **maneras de responderla**,
y cuál puedes usar depende de condiciones muy concretas del problema, no de
tu preferencia. Este notebook responde: **¿de qué depende que exista una
fórmula exacta, y qué hace scikit-learn cuando no existe?**

## 1. Las dos condiciones que necesita una fórmula cerrada

Recuerda de `00_introduccion_y_scikit_learn.ipynb`: para llegar a la fórmula
de mínimos cuadrados, tomamos las derivadas del costo, las igualamos a cero, y
**despejamos** $w$ y $b$ como quien despeja una incógnita en álgebra de
secundaria. Ese "despeje" solo es posible si se cumplen dos condiciones a la
vez:

1. **El modelo es lineal en sus parámetros.** $\hat y = wx + b$ lo es: $w$ y
   $b$ aparecen multiplicando o sumando, sin exponentes, sin funciones
   extrañas envolviéndolos. Una red neuronal, en cambio, compone muchas
   multiplicaciones y funciones no lineales (sigmoides, ReLU...) unas dentro
   de otras — no hay forma de "despejar" los parámetros de esa maraña.
2. **La función de costo es cuadrática (una parábola perfecta).** MSE lo es,
   porque eleva el residuo al cuadrado. Si en vez de MSE tu costo tiene una
   "esquina" (como MAE, $|r|$, en $r=0$) o involucra una función que no se
   puede invertir algebraicamente (como la sigmoide en la regresión
   logística), igualar la derivada a cero no da una ecuación que puedas
   despejar de forma directa.

Cuando **ambas** se cumplen — modelo lineal + costo cuadrático —, igualar las
derivadas a cero produce un sistema de ecuaciones *lineales*, y eso sí se
puede resolver siempre con álgebra (por eso `LinearRegression` nunca "falla"
ni necesita ajustar nada: la solución existe matemáticamente, siempre).
Apenas falta una de las dos condiciones, ya no hay atajo algebraico: hace
falta buscar el mínimo con un método iterativo — descenso de gradiente u
otro.

### 1.1 "No lineal en `x`" no es lo mismo que "no lineal en los parámetros"

Esta es la confusión más común con la condición 1, así que vale la pena un
ejemplo. Un modelo puede dibujar una curva (no una recta) y **seguir siendo
lineal en sus parámetros**. Compara estos dos:

- $\hat y = w_1 x + w_2 x^2 + b$ — esto dibuja una **parábola** en el plano
  $(x,\hat y)$, claramente no una recta. Pero míralo como función de
  $w_1, w_2, b$: siguen siendo términos que solo se *multiplican* por algo
  conocido ($x$ o $x^2$) y se *suman* — ningún parámetro está atrapado dentro
  de un exponente o una función. **Sigue siendo lineal en los parámetros.**
- $\hat y = \sigma(wx+b) = \dfrac{1}{1+e^{-(wx+b)}}$ (regresión logística) —
  aquí $w$ y $b$ quedan atrapados **dentro** de una exponencial. No hay forma
  de despejarlos con álgebra de sistemas lineales. **No es lineal en los
  parámetros.**

La consecuencia práctica es sorprendente: aunque el primer caso "se ve" más
complicado porque dibuja una curva, sigue teniendo fórmula cerrada — basta
con tratar $x^2$ como si fuera una variable de entrada más (una columna
extra), y aplicar exactamente la misma maquinaria de mínimos cuadrados que
usaste en `00_introduccion_y_scikit_learn.ipynb`. Compruébalo entrenando ese
modelo cuadrático con `LinearRegression` y mirando si aparece `n_iter_`:

In [1]:
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression

rng_poli = np.random.default_rng(1)
x_poli = np.linspace(-3, 3, 30)
y_poli = 2 * x_poli**2 - x_poli + 5 + rng_poli.normal(scale=1.0, size=30)

X_poli = np.column_stack([x_poli, x_poli**2])  # tratamos x y x² como dos variables de entrada
modelo_poli = LinearRegression().fit(X_poli, y_poli)

print(f"w1 (coeficiente de x):  {modelo_poli.coef_[0]:.3f}  (el dato real usa -1)")
print(f"w2 (coeficiente de x²): {modelo_poli.coef_[1]:.3f}  (el dato real usa 2)")
print(f"b:                       {modelo_poli.intercept_:.3f}  (el dato real usa 5)")
print(f"n_iter_: {getattr(modelo_poli, 'n_iter_', 'no existe este atributo')}")

w1 (coeficiente de x):  -1.123  (el dato real usa -1)
w2 (coeficiente de x²): 2.012  (el dato real usa 2)
b:                       4.894  (el dato real usa 5)
n_iter_: no existe este atributo


In [2]:
x_linea = np.linspace(-3, 3, 100)
y_linea = modelo_poli.predict(np.column_stack([x_linea, x_linea**2]))

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_poli, y=y_poli, mode="markers", name="datos", marker={"size": 9, "color": "black"}))
fig.add_trace(go.Scatter(x=x_linea, y=y_linea, mode="lines", name="ajuste (mínimos cuadrados, sin iterar)", line={"color": "#d62728", "width": 3}))
fig.update_layout(
    title="Aunque el ajuste dibuja una parábola, sigue siendo mínimos cuadrados exacto",
    xaxis_title="x", yaxis_title="y",
)
fig.show()

Los coeficientes recuperados (`w1≈-1.12`, `w2≈2.01`, `b≈4.89`) están muy cerca
de los valores reales (`-1`, `2`, `5`; la diferencia es solo el ruido que
agregamos a los datos), y `n_iter_` **no existe** — scikit-learn resolvió esto
con álgebra directa, exactamente igual que con una recta. La curva en el
gráfico no es señal de que haga falta iterar: lo único que importa es cómo
aparecen los parámetros en la ecuación, no la forma que dibuja `x`. Esta es
también la idea detrás de la regresión polinomial que verás más adelante
(fila "Features polinomiales" en [GLOSARIO.md](../../../../GLOSARIO.md)).

### 1.2 Por qué la sigmoide de la regresión logística no se puede despejar

Ahora el otro lado: un caso donde de verdad no hay fórmula cerrada, y se
puede ver exactamente por qué. En regresión logística, el mínimo del costo
(*log loss*) se alcanza cuando, para cada observación,
$\sigma(z) = y$, con $z = wx+b$ y $y$ la etiqueta real (0 o 1). El problema
aparece al intentar despejar $w$ y $b$ de ahí: la sigmoide **nunca llega
exactamente** a 0 ni a 1, solo se acerca cada vez más a medida que $z$ crece o
decrece sin límite.

In [3]:
z = np.linspace(-10, 10, 200)
sigmoide = 1 / (1 + np.exp(-z))

fig = go.Figure()
fig.add_trace(go.Scatter(x=z, y=sigmoide, mode="lines", name="σ(z)", line={"color": "#1f77b4", "width": 3}))
fig.add_hline(y=1, line_dash="dash", line_color="#d62728", annotation_text="y=1 (nunca se alcanza)")
fig.add_hline(y=0, line_dash="dash", line_color="#d62728", annotation_text="y=0 (nunca se alcanza)")
fig.update_layout(
    title="La sigmoide se acerca a 0 y 1 pero nunca los toca: ningún z finito resuelve σ(z)=y",
    xaxis_title="z = wx + b", yaxis_title="σ(z)",
)
fig.show()

Por eso no existe un valor finito de $w,b$ que resuelva exactamente esa
ecuación cuando las etiquetas son puros 0 y 1 — solo puedes *acercarte* cada
vez más, agrandando $w$ y $b$ sin límite. Ese es precisamente el trabajo del
optimizador iterativo de `LogisticRegression`: dar pasos hacia una mejor
aproximación, sabiendo de antemano que una solución exacta y finita ni
siquiera existe.

Esto no es solo una curiosidad matemática: es la raíz de un problema real
llamado **separación perfecta**. Si tus clases quedan completamente
separadas por alguna variable, el optimizador puede intentar agrandar los
coeficientes indefinidamente persiguiendo un óptimo que nunca llega — por eso
`LogisticRegression` regulariza por defecto (agrega una penalización, igual
que `Ridge`), en parte para frenar esa búsqueda infinita y devolver siempre
un `w,b` finito y razonable.

## 2. Cómo saber, sin adivinar, si un modelo usó fórmula cerrada o iteró

No hace falta memorizar cuáles modelos tienen fórmula cerrada: scikit-learn
lo delata. Todo modelo que ajusta sus parámetros **iterando** guarda, después
de `.fit()`, un atributo `n_iter_` con la cantidad de iteraciones que le tomó
converger. Un modelo que resuelve con álgebra directa, en cambio, no necesita
iterar — así que ese atributo no existe o queda en `None`.

Probemos con seis modelos sobre los mismos datos sintéticos (tres variables de
entrada, sin ningún caso difícil).

In [4]:
import numpy as np
import polars as pl
from sklearn.linear_model import (
    HuberRegressor,
    Lasso,
    LinearRegression,
    LogisticRegression,
    QuantileRegressor,
    Ridge,
)

rng = np.random.default_rng(0)
X = rng.normal(size=(50, 3))
y = X @ np.array([1.5, -2.0, 0.5]) + rng.normal(scale=0.1, size=50)
y_clase = (y > np.median(y)).astype(int)

modelos = {
    "LinearRegression (MSE)": LinearRegression(),
    "Ridge (MSE + L2)": Ridge(),
    "Lasso (MSE + L1)": Lasso(),
    "HuberRegressor (Huber)": HuberRegressor(),
    "QuantileRegressor (MAE, solver='highs')": QuantileRegressor(solver="highs"),
    "LogisticRegression (log loss)": LogisticRegression(),
}

filas = []
for nombre, modelo in modelos.items():
    if nombre == "LogisticRegression (log loss)":
        modelo.fit(X, y_clase)
    else:
        modelo.fit(X, y)
    filas.append({"modelo": nombre, "n_iter_": str(getattr(modelo, "n_iter_", "no existe este atributo"))})

pl.DataFrame(filas)

modelo,n_iter_
str,str
"""LinearRegression (MSE)""","""no existe este atributo"""
"""Ridge (MSE + L2)""","""None"""
"""Lasso (MSE + L1)""","""4"""
"""HuberRegressor (Huber)""","""17"""
"""QuantileRegressor (MAE, solver…","""50"""
"""LogisticRegression (log loss)""","""[7]"""


El patrón es exactamente el que predice la teoría de la sección 1:

- **`LinearRegression`**: el atributo `n_iter_` **ni siquiera existe** —
  scikit-learn no tiene nada que contar, porque nunca iteró. Modelo lineal +
  MSE: cumple las dos condiciones.
- **`Ridge`**: existe el atributo, pero vale `None`. Sigue siendo modelo
  lineal + MSE (la penalización L2 no rompe la forma cuadrática), así que con
  este tamaño de datos scikit-learn eligió automáticamente un solver cerrado.
- **`Lasso`**: `n_iter_ = 4`. Aunque el modelo sigue siendo lineal, la
  penalización L1 tiene una esquina (igual que `|r|` en MAE) — ya no es
  cuadrática pura, así que hace falta un método iterativo (descenso por
  coordenadas).
- **`HuberRegressor`**: `n_iter_ = 17`. Modelo lineal, pero Huber Loss no es
  una parábola en todas partes (recuerda el tramo lineal fuera de $\pm\delta$
  de `03_huber_loss_en_profundidad.ipynb`) — tampoco hay fórmula cerrada.
- **`QuantileRegressor`**: `n_iter_ = 50`. Como viste en
  [`05_mae_en_profundidad.ipynb`](05_mae_en_profundidad.ipynb), la esquina de
  MAE en `r=0` ni siquiera admite descenso de gradiente clásico — estas
  "iteraciones" son del solver de programación lineal, una familia de método
  completamente distinta.
- **`LogisticRegression`**: `n_iter_ = [7]`. Aquí falla la *otra* condición:
  el costo (log loss) involucra una sigmoide que no se puede despejar
  algebraicamente, sin importar que el modelo sea lineal por dentro.

## 3. Panorama general

| Situación | ¿Modelo lineal en sus parámetros? | ¿Costo cuadrático? | ¿Fórmula cerrada? | Qué usa scikit-learn |
| --- | --- | --- | --- | --- |
| `LinearRegression` (MSE) | Sí | Sí | Sí | Mínimos cuadrados (ecuaciones normales / SVD) |
| `Ridge` (MSE + L2) | Sí | Sí | Sí | Mínimos cuadrados regularizados |
| `Lasso` (MSE + L1) | Sí | No (esquina en la penalización) | No | Descenso por coordenadas |
| `LogisticRegression` (log loss) | Sí | No (sigmoide no invertible) | No | Optimizadores iterativos (`lbfgs`, `saga`...) |
| `HuberRegressor` (Huber) | Sí | No (tramo lineal fuera de $\delta$) | No | Optimización iterativa (`lbfgs`) |
| `QuantileRegressor` (MAE) | Sí | No (esquina en $r=0$) | No | Programación lineal |
| Redes neuronales (cualquier costo) | No | — | No, nunca | Descenso de gradiente (backpropagation) |

Un matiz práctico que no aparece en la tabla: incluso cuando **sí** existe
fórmula cerrada, si el dataset tiene muchísimas filas o variables, invertir la
matriz que exige esa fórmula puede volverse costoso. Ahí, por conveniencia
computacional (no porque la fórmula deje de existir), a veces se prefiere un
método iterativo como `SGDRegressor(loss="squared_error")` — la única
situación donde se elige descenso de gradiente teniendo mínimos cuadrados
disponible.

## 4. Ideas clave

- La función de costo (MSE, log loss, Huber...) es la pregunta; mínimos
  cuadrados y descenso de gradiente son formas de responderla. Cuál puedes
  usar depende del problema, no es una elección de estilo.
- Una fórmula cerrada exige **dos** condiciones a la vez: modelo lineal en
  sus parámetros y costo cuadrático. Si falta cualquiera de las dos, hace
  falta un método iterativo.
- No necesitas memorizar qué modelo usa qué: el atributo `n_iter_` te lo dice
  directamente — ausente o `None` significa fórmula cerrada; un número
  significa que iteró.
- "Iterativo" no siempre significa "descenso de gradiente": Lasso usa
  descenso por coordenadas, MAE usa programación lineal, y cada método existe
  porque la forma de esa pérdida en particular lo requiere.

**Ejercicio:** cambia el `solver` de `Ridge` a uno explícitamente iterativo,
por ejemplo `Ridge(solver="sag")`, y vuelve a ejecutar la sección 2. Antes de
ejecutar, predice: ¿`n_iter_` va a seguir siendo `None`, o ahora va a mostrar
un número? ¿Cambia esto algo de lo que dijimos en la sección 1 sobre si Ridge
"tiene" o "no tiene" fórmula cerrada?